In [12]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md
from sklearn.metrics import mean_absolute_error   # la MAE, pour plus tard
import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mrdwin/groupe5-allstate-appautomatque-l2-esgcvak/a_estimer.csv
/kaggle/input/datasets/mrdwin/groupe5-allstate-appautomatque-l2-esgcvak/train.csv


Séance : 1
Nom : HOUNSOU Joamour Cadmon Bedwin
Porteur : P1
Etapes assignés : 1.1 ---> 1.8 --> 1.7 --> 1.11

In [9]:
!ls /kaggle/input/

datasets


In [13]:
DOSSIER = "/kaggle/input/datasets/mrdwin/groupe5-allstate-appautomatque-l2-esgcvak"

print(os.listdir(DOSSIER))

train = pd.read_csv(f"{DOSSIER}/train.csv")
a_estimer = pd.read_csv(f"{DOSSIER}/a_estimer.csv")

print("train     :", train.shape)
print("a_estimer :", a_estimer.shape)

['a_estimer.csv', 'train.csv']
train     : (150654, 132)
a_estimer : (37664, 131)


In [15]:
#pourquoi je dois commencer a inventorier ? il est important d'inventorier pour vérifier si tout les fichiers sont aucomplets et qu'on travaille sur les bonnes donnes , il nous permetr d'indentifier ou se trouve le loss

cols_cat  = [c for c in train.columns if c.startswith("cat")]
cols_cont = [c for c in train.columns if c.startswith("cont")]
autres    = [c for c in train.columns if c not in cols_cat + cols_cont]

print(len(cols_cat), "catégorielles |", len(cols_cont), "continues")
print("autres colonnes :", autres)
print(train.dtypes.value_counts())

print("absentes de a_estimer :", set(train.columns) - set(a_estimer.columns))
print("absentes de train     :", set(a_estimer.columns) - set(train.columns))

#ensuite je vérifie les valeurs manquantes 
print("manquants train     :", int(train.isna().sum().sum()))
print("manquants a_estimer :", int(a_estimer.isna().sum().sum()))

#comme je ne sais pas a quoi ressembles les variables il faut que je les identifient 
nb_modalites = train[cols_cat].nunique().sort_values(ascending=False)
print(nb_modalites.head(5))
print("min :", nb_modalites.min(), "| max :", nb_modalites.max())

# modalités présentes dans a_estimer mais jamais vues dans train
nouvelles = {c: set(a_estimer[c]) - set(train[c]) for c in cols_cat}
nouvelles = {c: v for c, v in nouvelles.items() if v}
print(len(nouvelles), "variables ont des modalités absentes du train")

train[cols_cont].describe().T[["min", "max", "mean"]]
#maintenant a quoi ressemble la cible 
print(train["loss"].describe())

inventaire = pd.DataFrame({
    "table":       ["train", "a_estimer"],
    "lignes":      [len(train), len(a_estimer)],
    "colonnes":    [train.shape[1], a_estimer.shape[1]],
    "manquants":   [int(train.isna().sum().sum()), int(a_estimer.isna().sum().sum())],
    "contient loss": ["loss" in train.columns, "loss" in a_estimer.columns],
})
inventaire

116 catégorielles | 14 continues
autres colonnes : ['id', 'loss']
object     116
float64     16
Name: count, dtype: int64
absentes de a_estimer : {'loss'}
absentes de train     : set()
manquants train     : 0
manquants a_estimer : 0
cat116    317
cat110    130
cat109     80
cat113     61
cat112     51
dtype: int64
min : 2 | max : 317
5 variables ont des modalités absentes du train
count    150654.000000
mean       3038.922785
std        2915.009036
min           0.670000
25%        1204.750000
50%        2115.570000
75%        3865.632500
max      121012.250000
Name: loss, dtype: float64


,table,lignes,colonnes,manquants,contient loss
0,train,150654,132,0,True
1,a_estimer,37664,131,0,False


In [16]:
#DOUBLONS 
print("id dupliqués dans train     :", train["id"].duplicated().sum())
print("id dupliqués dans a_estimer :", a_estimer["id"].duplicated().sum())
print("id communs aux deux tables  :", len(set(train["id"]) & set(a_estimer["id"])))

variables = cols_cat + cols_cont

identiques_vars = train.duplicated(subset=variables, keep=False).sum()
identiques_tout = train.duplicated(subset=variables + ["loss"], keep=False).sum()

print("lignes de train aux variables identiques à une autre ligne :", identiques_vars)
print("... dont le montant (loss) est aussi identique            :", identiques_tout)
print("lignes dupliquées dans a_estimer :", a_estimer.duplicated(subset=variables).sum())

empreintes_train = pd.util.hash_pandas_object(train[variables], index=False)
empreintes_test  = pd.util.hash_pandas_object(a_estimer[variables], index=False)

recouvrement = empreintes_test.isin(set(empreintes_train)).sum()
print("lignes de a_estimer dont les variables existent déjà dans train :", recouvrement)

id dupliqués dans train     : 0
id dupliqués dans a_estimer : 0
id communs aux deux tables  : 0
lignes de train aux variables identiques à une autre ligne : 2
... dont le montant (loss) est aussi identique            : 0
lignes dupliquées dans a_estimer : 0
lignes de a_estimer dont les variables existent déjà dans train : 0


In [17]:
#FUITES
print("loss présente dans a_estimer :", "loss" in a_estimer.columns)
print("id train     : de", train["id"].min(), "à", train["id"].max())
print("id a_estimer : de", a_estimer["id"].min(), "à", a_estimer["id"].max())
print("lien id / loss (Spearman) :", round(train["id"].corr(train["loss"], method="spearman"), 4))

lien_cont = (train[cols_cont]
             .apply(lambda s: s.corr(train["loss"], method="spearman"))
             .abs().sort_values(ascending=False))
print(lien_cont.head(5))

y = np.log1p(train["loss"])

def part_expliquee(col):
    # part de la variation de y expliquée par les seules modalités de la colonne (0 à 1)
    moyennes = y.groupby(train[col]).transform("mean")
    return ((moyennes - y.mean()) ** 2).sum() / ((y - y.mean()) ** 2).sum()

lien_cat = pd.Series({c: part_expliquee(c) for c in cols_cat}).sort_values(ascending=False)
print(lien_cat.head(8))

comparaison = pd.DataFrame({"moy_train": train[cols_cont].mean(),
                            "moy_a_estimer": a_estimer[cols_cont].mean()})
comparaison["ecart_%"] = ((comparaison["moy_a_estimer"] - comparaison["moy_train"])
                          / comparaison["moy_train"].abs() * 100).round(1)
comparaison

loss présente dans a_estimer : False
id train     : de 1.0 à 587633.0
id a_estimer : de 5.0 à 587630.0
lien id / loss (Spearman) : -0.0033
cont2     0.080134
cont3     0.068636
cont7     0.058375
cont11    0.056516
cont12    0.056043
dtype: float64
cat80     0.237275
cat79     0.175518
cat101    0.146848
cat100    0.136388
cat87     0.115019
cat12     0.114357
cat10     0.089223
cat81     0.086335
dtype: float64


,moy_train,moy_a_estimer,ecart_%
cont1,0.493961,0.493463,-0.1
cont2,0.507243,0.506971,-0.1
cont3,0.498806,0.499369,0.1
cont4,0.491315,0.493801,0.5
cont5,0.487207,0.488312,0.2
cont6,0.491086,0.490381,-0.1
cont7,0.485005,0.484831,-0.0
cont8,0.485801,0.488984,0.7
cont9,0.485533,0.485400,-0.0
cont10,0.498250,0.497330,-0.2


In [18]:
# ===== LA MÉTRIQUE : MAE (erreur absolue moyenne, en euros) =====
y = train["loss"]

# 1. Les deux constantes de référence (une seule valeur annoncée pour tous les sinistres)
moyenne, mediane = y.mean(), y.median()
mae_moyenne = mean_absolute_error(y, np.full(len(y), moyenne))
mae_mediane = mean_absolute_error(y, np.full(len(y), mediane))

print(f"moyenne = {moyenne:,.0f} -> MAE = {mae_moyenne:,.2f}")
print(f"médiane = {mediane:,.0f} -> MAE = {mae_mediane:,.2f}")
print(f"écart : la médiane fait {(mae_moyenne - mae_mediane) / mae_moyenne:.1%} mieux que la moyenne")

# 2. Preuve : la MAE est minimale à la médiane (on teste 91 constantes)
grille = np.quantile(y, np.linspace(0.05, 0.95, 91))
mae_grille = [np.mean(np.abs(y - c)) for c in grille]
meilleure = grille[int(np.argmin(mae_grille))]
print(f"\nconstante qui minimise la MAE : {meilleure:,.0f} (médiane : {mediane:,.0f})")

# 3. Coût asymétrique : sous-estimer coûte k fois plus que surestimer
print("\nk | quantile | constante optimale | coût moyen")
for k in (1, 2, 3, 5):
    q = k / (1 + k)
    c = np.quantile(y, q)
    ecart = y - c                                    # > 0 : on a sous-estimé
    cout = np.mean(np.where(ecart > 0, k * ecart, -ecart))
    print(f"{k} | {q:.2f}     | {c:>10,.0f}         | {cout:,.1f}")

moyenne = 3,039 -> MAE = 1,968.14
médiane = 2,116 -> MAE = 1,810.49
écart : la médiane fait 8.0% mieux que la moyenne

constante qui minimise la MAE : 2,116 (médiane : 2,116)

k | quantile | constante optimale | coût moyen
1 | 0.50     |      2,116         | 1,810.5
2 | 0.67     |      3,099         | 2,951.4
3 | 0.75     |      3,866         | 3,794.3
5 | 0.83     |      4,954         | 5,026.5
